# Projet 4 — Partie 2 : Conteneurisation avec Docker

**Projet DataSoluTech — MongoDB et Python**

Ce notebook documente la deuxième partie du projet : rendre notre solution de migration **reproductible avec Docker**, sécuriser les accès à MongoDB et vérifier son fonctionnement.

La **partie 1**, dans `P4_analyse_dataset_medical.ipynb`, se termine avec les tests **CRUD**. Nous ne refaisons pas ici l'analyse détaillée du CSV.

> **Confidentialité :** ne pas enregistrer dans ce notebook de mots de passe, de noms de patients ou de résultats contenant des documents médicaux individuels. Le fichier CSV et le fichier `.env` ne doivent pas être publiés sur GitHub.

## 1. Point de départ et objectifs

À l'issue de la première partie :

- Le CSV comporte **55 500 lignes et 15 colonnes**.
- **534 doublons exacts** ont été supprimés selon la règle retenue pour le projet.
- Il reste **54 966 hospitalisations** à migrer.
- Les documents MongoDB sont organisés en quatre sous-documents : `patient`, `medical`, `hospitalisation` et `facturation`.
- La migration initiale et les opérations CRUD ont été testées avec Python et MongoDB en local.

**Objectifs de la partie 2 :**

1. Exécuter MongoDB dans un conteneur Docker.
2. Exécuter également `migration.py` dans un conteneur Python.
3. Conserver les données entre deux démarrages.
4. Sécuriser les accès avec trois profils utilisateurs.
5. Documenter les vérifications et versionner les fichiers utiles sur GitHub.

Les essais décrits ci-dessous ont été réalisés sur **Windows 11**, avec Docker Desktop et PowerShell.

## 2. Quelques notions Docker

- **Image** : modèle utilisé pour construire un conteneur (par exemple `mongo:8.0` ou une image Python 3.12).
- **Conteneur** : environnement isolé qui exécute un service ou un script.
- **Dockerfile** : fichier décrivant comment construire notre image Python.
- **Docker Compose** : fichier YAML qui définit les services et leurs connexions.
- **Volume** : stockage séparé du cycle de vie du conteneur, utilisé pour conserver les données MongoDB.

Notre architecture :

```text
CSV local (data/healthcare_dataset.csv)
            |
            | volume en lecture seule
            v
   Conteneur Python (migration)
            |
            | réseau Docker : mongodb:27017
            v
    Conteneur MongoDB (mongodb)
            |
            v
     Volume nommé mongo_data
```

Depuis **Windows**, nous accédons à ce MongoDB par `127.0.0.1:27018`. Entre les **conteneurs**, Python utilise `mongodb:27017`. Cela évite un conflit avec MongoDB installé directement sur Windows, qui utilise déjà le port `27017`.

## 3. Organisation des fichiers

Les principaux fichiers de la partie Docker se trouvent à la racine du projet :

```text
Project_4/
├── data/
│   └── healthcare_dataset.csv    # local, exclu de Git
├── migration.py                  # script Python de migration
├── Dockerfile                    # image du script Python
├── docker-compose.yml            # orchestration des 2 services
├── requirements.txt              # dépendances Python versionnées
├── .env                          # identifiants, exclu de Git
├── .dockerignore                 # exclusions lors du build
├── .gitignore                    # exclusions Git
├── P4_analyse_dataset_medical.ipynb
└── P4_conteneurisation_docker.ipynb
```

L'exécution dans Docker **ne remplace pas** le notebook : `migration.py` permet une migration automatisée et le notebook explique les choix et les essais.

## 4. Dépendances Python — `requirements.txt`

Les bibliothèques utilisées par le script sont :

| Bibliothèque | Utilité |
|---|---|
| `pandas` | Charger, nettoyer et transformer le CSV |
| `pymongo` | Communiquer avec MongoDB |
| `python-dotenv` | Lire les variables du fichier `.env` lors d'une exécution locale |

Nous avons créé `requirements.txt` avec les **versions réellement installées** sur notre machine, au format `pandas==X.Y.Z`.

Commande utilisée dans PowerShell pour connaître les versions :

```powershell
python -m pip freeze | Select-String "^(pandas|pymongo|python-dotenv)=="
```

`os` et `pathlib` font partie de Python ; ils n'ont pas besoin de figurer dans `requirements.txt`.

## 5. Construire l'image Python — `Dockerfile`

Notre Dockerfile est simple :

```dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY migration.py .

CMD ["python", "migration.py"]
```

**Explication :**

- `FROM` : choisit une image Python légère.
- `WORKDIR` : définit le répertoire de travail du conteneur.
- `COPY requirements.txt` + `RUN` : installent les dépendances.
- `COPY migration.py` : ajoute le script à l'image.
- `CMD` : précise la commande exécutée au lancement.

Le CSV n'est **pas inclus dans l'image** : Docker Compose le rendra accessible par un volume monté en lecture seule.

## 6. Protéger les fichiers sensibles

Dans `.gitignore`, nous excluons notamment :

```gitignore
data/
.env
.env.*
!.env.example
.ipynb_checkpoints/
__pycache__/
.venv/
```

Dans `.dockerignore`, nous excluons le contexte inutile ou sensible :

```gitignore
.git/
data/
.env
.env.*
.ipynb_checkpoints/
*.ipynb
reports/
__pycache__/
```

Le fichier `.env` contient les identifiants (créés localement) :

```dotenv
MONGO_ROOT_USER=admin
MONGO_ROOT_PASSWORD=<mot_de_passe_local_non_publie>
```

**Ne jamais copier un véritable mot de passe dans le notebook ou sur GitHub.** Pour montrer les variables attendues dans le dépôt, nous pourrons créer un `.env.example` avec des valeurs fictives.

## 7. Déclarer les deux services — `docker-compose.yml`

Voici la configuration de référence utilisée pendant cette partie (à comparer avec le fichier présent dans le projet) :

```yaml
services:
  mongodb:
    image: mongo:8.0
    container_name: datasolutech_mongodb
    ports:
      - "127.0.0.1:27018:27017"
    environment:
      MONGO_INITDB_ROOT_USERNAME: ${MONGO_ROOT_USER}
      MONGO_INITDB_ROOT_PASSWORD: ${MONGO_ROOT_PASSWORD}
    volumes:
      - mongo_data:/data/db
    healthcheck:
      test: ["CMD", "mongosh", "--quiet", "--eval", "db.adminCommand({ ping: 1 })"]
      interval: 10s
      timeout: 5s
      retries: 5
      start_period: 20s

  migration:
    build: .
    depends_on:
      mongodb:
        condition: service_healthy
    environment:
      MONGO_HOST: mongodb
      MONGO_PORT: 27017
      MONGO_ROOT_USER: ${MONGO_ROOT_USER}
      MONGO_ROOT_PASSWORD: ${MONGO_ROOT_PASSWORD}
    volumes:
      - ./data:/app/data:ro

volumes:
  mongo_data:
```

**À retenir :**

- `mongo_data:/data/db` conserve les documents si le conteneur est recréé.
- `./data:/app/data:ro` donne accès au CSV **en lecture seule**.
- `depends_on` avec `service_healthy` attend un MongoDB considéré comme prêt.
- Le `healthcheck` teste la disponibilité de MongoDB, **pas les droits de connexion des utilisateurs**.

> Dans notre version actuelle, le conteneur Python reçoit encore les identifiants administrateur. Nous changerons cela pour le compte `data_engineer` dans la prochaine étape de sécurisation.

## 8. Démarrage et contrôles Docker

Commandes utilisées dans PowerShell, depuis la racine de `Project_4` :

```powershell
docker compose config --quiet
docker compose up -d mongodb
docker compose ps
```

- `config --quiet` : vérifie la syntaxe de Compose.
- `up -d mongodb` : lance MongoDB en arrière-plan.
- `ps` : affiche l'état des conteneurs.

Lors du premier lancement, Docker a téléchargé l'image MongoDB, créé le volume `project_4_mongo_data` et exposé le port `27018` sur Windows.

Après ajout du `healthcheck`, nous avons recréé le conteneur pour appliquer la configuration :

```powershell
docker compose up -d --force-recreate mongodb
docker compose ps
```

L'état est d'abord **`health: starting`**, puis doit passer à **`healthy`** si le contrôle réussit.

Si l'état reste anormal, nous pouvons consulter :

```powershell
docker compose logs mongodb
docker inspect datasolutech_mongodb --format '{{json .Config.Healthcheck}}'
```

Une sortie `null` de la deuxième commande signifie qu'aucun `healthcheck` n'a été intégré au conteneur.

## 9. Vérifier MongoDB à l'intérieur du conteneur

Dans **PowerShell** :

```powershell
docker exec -it datasolutech_mongodb mongosh -u admin -p --authenticationDatabase admin
```

Le paramètre `-p` demande le mot de passe sans l'inclure dans la commande.

Une fois connecté à `mongosh` :

```javascript
db.adminCommand({ ping: 1 })
```

**Résultat attendu :** `{ ok: 1 }`.

Ce test a permis de confirmer que MongoDB répond et que le compte administrateur est utilisable. Nous avons ensuite quitté le shell avec `exit`.

## 10. Tester la connexion Python → MongoDB Docker

Sur **Windows**, le serveur MongoDB du conteneur répond à l'adresse `localhost:27018`.

La cellule suivante teste uniquement le `ping` ; elle ne lit ni ne modifie les données médicales. Le fichier `.env` doit se trouver dans le dossier depuis lequel Jupyter exécute le notebook.

In [ ]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv(".env")

client_docker = MongoClient(
    "mongodb://localhost:27018/",
    username=os.getenv("MONGO_ROOT_USER"),
    password=os.getenv("MONGO_ROOT_PASSWORD"),
    authSource="admin"
)

try:
    client_docker.admin.command("ping")
    print("Connexion Python → MongoDB Docker : OK")
finally:
    client_docker.close()

## 11. Adapter `migration.py` aux deux environnements

Nous avons remplacé l'adresse MongoDB fixe par des paramètres lus dans les variables d'environnement :

```python
load_dotenv(".env")

host = os.getenv("MONGO_HOST", "localhost")
port = int(os.getenv("MONGO_PORT", "27018"))

client = MongoClient(
    host=host,
    port=port,
    username=os.getenv("MONGO_ROOT_USER"),
    password=os.getenv("MONGO_ROOT_PASSWORD"),
    authSource="admin"
)
```

Ainsi :

| Exécution | Adresse cible |
|---|---|
| Script exécuté sur Windows | `localhost:27018` |
| Script exécuté dans le conteneur `migration` | `mongodb:27017` |

Le **reste du script** continue à charger le CSV, supprimer les doublons, convertir les dates, construire les documents et les importer dans `healthcare.hospitalisations`.

La configuration montrée ici correspond à notre **étape intermédiaire** avec le compte admin. Elle sera remplacée par une connexion `data_engineer`.

## 12. Exécuter la migration dans Docker

Commande PowerShell utilisée :

```powershell
docker compose run --rm --build migration
```

- `--build` : construit ou reconstruit l'image Python.
- `run migration` : exécute le script dans un conteneur temporaire.
- `--rm` : supprime ce conteneur une fois le script terminé.

**Résultat de nos essais :** **54 966 documents** ont été migrés dans MongoDB Docker. Une exécution ultérieure a reconnu que la collection contenait déjà le nombre attendu et a évité une insertion supplémentaire.

Le script effectue aussi les contrôles de **nombre de documents** et de **structure**. Attention : un simple total de documents égal à 54 966 ne prouve pas, à lui seul, que toutes les valeurs sont identiques à la source.

Le conteneur Python est un **travail de migration ponctuel** : il n'a pas besoin de rester en fonctionnement permanent.

## 13. Sécuriser les accès : trois utilisateurs MongoDB

Nous avons choisi trois profils avec des permissions adaptées :

| Profil | Utilisateur | Rôle MongoDB | Base concernée |
|---|---|---|---|
| Administrateur | `admin` | `root` | Administration du serveur |
| Analyste | `analyste` | `read` | `healthcare` |
| Data Engineer | `data_engineer` | `readWrite` | `healthcare` |

**Pourquoi ?** Un analyste doit consulter les données sans pouvoir les modifier ; le Data Engineer doit pouvoir exécuter la migration ; seul l'administrateur doit gérer les utilisateurs.

Le compte `admin` a été créé lors de l'initialisation du conteneur avec les variables `MONGO_INITDB_ROOT_*`. Les deux autres comptes ont été créés manuellement dans la base `healthcare`.

> Les variables `MONGO_INITDB_ROOT_*` initialisent le compte administrateur **lors de la première création des données**. Modifier ces variables ne réinitialise pas le mot de passe d'un compte déjà stocké dans un volume existant.

## 14. Création des rôles `read` et `readWrite`

Après connexion administrateur avec `mongosh`, nous avons sélectionné la base :

```javascript
use healthcare
```

Puis créé les utilisateurs (les mots de passe sont saisis de façon interactive) :

```javascript
db.createUser({
    user: "analyste",
    pwd: passwordPrompt(),
    roles: [{ role: "read", db: "healthcare" }]
})

db.createUser({
    user: "data_engineer",
    pwd: passwordPrompt(),
    roles: [{ role: "readWrite", db: "healthcare" }]
})
```

Pour vérifier les utilisateurs de cette base :

```javascript
db.getUsers()
```

Les comptes sont stockés dans le volume MongoDB avec leurs rôles. **Un déploiement neuf nécessitera de recréer ces comptes** : leur automatisation est prévue pour une prochaine étape.

## 15. Vérifier les permissions sans toucher aux hospitalisations

Nous avons utilisé une collection distincte, `tests_permissions`, pour les essais.

**Connexion analyste depuis PowerShell :**

```powershell
docker exec -it datasolutech_mongodb mongosh -u analyste -p --authenticationDatabase healthcare
```

Dans `mongosh` :

```javascript
use healthcare
db.hospitalisations.countDocuments({})
db.tests_permissions.insertOne({ test: "analyste" })
```

Le comptage doit fonctionner ; l'insertion doit échouer avec une erreur d'autorisation.

**Connexion Data Engineer :**

```powershell
docker exec -it datasolutech_mongodb mongosh -u data_engineer -p --authenticationDatabase healthcare
```

Les opérations `insertOne`, `findOne`, `updateOne` et `deleteOne` ont été testées sur un document temporaire de `tests_permissions`, puis ce document a été supprimé.

Cela vérifie que l'analyste est en **lecture seule** et que le Data Engineer dispose des droits **CRUD** sur `healthcare`.

## 16. Modifier les mots de passe sans changer les droits

Depuis une session `mongosh` administrateur, dans `healthcare` :

```javascript
use healthcare
db.changeUserPassword("analyste", passwordPrompt())
db.changeUserPassword("data_engineer", passwordPrompt())
```

Ces commandes permettent de changer les mots de passe sans recréer les utilisateurs, sans modifier leurs rôles et sans toucher aux documents médicaux.

Les mots de passe ne doivent apparaître **ni dans GitHub ni dans le notebook**.

## 17. Bilan de la conteneurisation

| Vérification | Bilan des essais réalisés |
|---|---|
| Démarrage du conteneur MongoDB | Réussi |
| Volume persistant `mongo_data` | Créé et utilisé |
| Authentification administrateur | Testée |
| Connexion Python à MongoDB Docker | Réussie |
| Script Python exécuté dans Docker | Réussi |
| Documents migrés | **54 966** |
| Vérifications de nombre et de structure | Réussies |
| Healthcheck Docker | Configuré ; état de démarrage observé |
| Trois profils utilisateurs | Définis et créés |
| Permissions `read` / `readWrite` | Testées |

**Limite actuelle :** Docker Compose rend l'environnement reproductible, mais ne réalise pas à lui seul un déploiement MongoDB distribué avec scalabilité horizontale.

## 18. Suite du travail

Il reste à :

1. Modifier `migration.py` et `docker-compose.yml` pour que **la migration utilise `data_engineer`**, pas `admin`.
2. Automatiser la création des comptes `analyste` et `data_engineer` lors d'un **déploiement neuf**.
3. Compléter les tests automatisés et la documentation de réinstallation.
4. Versionner les nouveaux fichiers (`Dockerfile`, Compose, `requirements.txt`, ce notebook, etc.) sur GitHub sans envoyer `data/` ni `.env`.
5. Préparer l'étude comparative des services AWS, puis la présentation finale.

**Important pour GitHub :** vider les sorties des cellules Jupyter avant le commit, et vérifier `git status` pour ne publier aucun fichier médical ou secret.